In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [ ]:
len(documents)

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
doc = documents[0]
doc
print(doc["content"])
print(doc["filename"])


In [ ]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [ ]:
import json

user_prompt = json.dumps(doc)

user_prompt

In [ ]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [ ]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [ ]:
result = response.output_parsed

print(result)

In [ ]:
print(result.questions)

In [ ]:
from evaluation_utils import llm_structured

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

In [ ]:
print(usage)

In [ ]:
from evaluation_utils import calc_price

cost = calc_price(usage)

cost

In [ ]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

In [ ]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

In [ ]:
generate_ground_truth(doc)

In [ ]:
# Question 1
target_filenames = {
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
}

generated_records = []

for doc in documents:
    if doc["filename"] in target_filenames:
        ground_truth, usage = generate_ground_truth(doc)
        generated_records.extend(ground_truth)
        print(doc["filename"], usage)

generated_records

In [ ]:
import pandas as pd

# Load CSV into a DataFrame
df = pd.read_csv("data/ground-truth.csv")

# Convert DataFrame rows into a list of dictionaries
ground_truth = df.to_dict(orient="records")

In [ ]:
print(ground_truth[0])

In [ ]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

len(chunks)

In [ ]:
chunks[0]

In [ ]:
from embedder import Embedder

embed = Embedder()

In [ ]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = [chunk["filename"] for chunk in chunks[i:i + batch_size]]
    batch_vectors = embed.encode_batch(batch)
    vectors.extend(batch_vectors)

len(vectors)

In [ ]:
import numpy as np
X = np.array(vectors)

X.shape

In [ ]:
q = ground_truth[0]["question"]

query_vector = embed.encode(q)


In [ ]:
def vector_search(query, num_results=10):
    from minsearch import VectorSearch

    vindex = VectorSearch(keyword_fields=["course"])
    vindex.fit(X, chunks)

    results = vindex.search(query,
                         num_results=5)

    results
    return results

In [ ]:
def text_search(query, num_results=10):
    from minsearch import Index

    index = Index(
        text_fields=['content'],
        keyword_fields=['course']
    )
    index.fit(chunks)

    search_results = index.search(
        query,
        boost_dict={"question": 2.0, "section": 0.5},
        num_results=num_results
    )

    search_results

    return search_results

In [ ]:
question = ground_truth[0]["question"]

t_filenames = text_search(question, num_results=5)
t_filenames

In [ ]:
question = ground_truth[0]["question"]
query_vector = embed.encode(question)

t_filenames = vector_search(query_vector, num_results=5)
t_filenames

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
question = "What exactly is a retrieval-augmented generation system, and why does it help with answers that the model wouldn't know on its own?"
hybrid_search(question, k=60)

In [ ]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground-truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [ ]:
q = ground_truth[0]
q

In [ ]:
doc_id = q["filename"]
text_results = text_search(query=q["question"],num_results=5)

In [ ]:
text_results

In [ ]:
for d in text_results:
    print(f'{d["filename"]} == {doc_id}: {d["filename"] == doc_id}')

In [ ]:
doc_id = q["filename"]
query_vector = embed.encode(q["question"])
vector_results = vector_search(query_vector,num_results=5)



In [ ]:
vector_results

In [ ]:
for d in vector_results:
    print(f'{d["filename"]} == {doc_id}: {d["filename"] == doc_id}')

In [ ]:
relevance = []

for d in results:
    relevance.append(int(d["filename"] == doc_id))

relevance

In [ ]:
def compute_relevance_text(q):
    doc_id = q["filename"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [ ]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total_text = compute_relevance_total_text(ground_truth)



In [ ]:
relevance_total_text

In [ ]:
def compute_relevance(q, search_function):
    doc_id = q["filename"]
    # add if else based on search function..text and vector
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["filename"] == doc_id))

    return relevance

In [ ]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [ ]:
relevance_total = compute_relevance_total(ground_truth, text_search)

In [ ]:
relevance_total

In [ ]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [ ]:
# Hit rate with text search
hit_rate(relevance_total)

In [281]:
relevance_total = compute_relevance_total(ground_truth, vector_search)

  0%|          | 0/360 [00:00<?, ?it/s]

AttributeError: 'str' object has no attribute 'reshape'

In [ ]:
relevance_total